In [2]:
#This is a clean document trying to break down the density calculation, does it make sense? 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from scipy import stats
from scipy.optimize import minimize
from sklearn.utils import resample

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

print('Imports successful')

Imports successful



## Simple approach to  the density model

If we just use the mean values of the parameters (cT, cW, cR, cH, Tc), reported in Winterl et al. 2024 and plug them in the windchill formula:

rho_pred = rho_max/(1 + exp( cT*T + cW*W + cH*H + cR*R - Tc ))  

with:
rho_max = 12.83 1/m²
T : temperature in K, W: windspeed in m/s, H: relative humidity in %, R: solar radiation in W/m²

Then use rho_pred and area A to predict the number of individuals

count_pred = rho_pred * A



###  Supplementary Table 3 (Winterl et al. 2024)
The table contains the numerical values for the best estimate of the windchill model
parameters.  

|      |   cT     |    cW     |     cR       |     cH    |       Tc     |      sigma |
|------|----------|-----------|--------------|-----------|--------------|------------| 
| mean |-0.023845 | 0.079986  | -0.000767    | 0.008434  |   -4.903825  |    0.731349| 
| std  | 0.003476 |  0.007152 |   0.000133   | 0.001784  |    0.909726  |    0.023509|


In [ ]:
#Original 
cT = -0.023845
cW = 0.079986 
cR = -0.000767 
cH = 0.008434 
Tc = -4.903825 
sigma = 0.731349
rho_max = 12.83

# Posterior SDs from convergence diagnostics table
sd_T  = 0.003476
sd_W  = 0.007152
sd_R  = 0.000133
sd_H  = 0.001784
sd_Tc = 0.909726

# One example from Amanda Bay - first row of the dfRFD csv file AMAN 139
# Windspeed         Temperature (K)     Humidity            Radiation
#3.380879163742065	247.7754821777344	74.00595786344412	0

T = 247.7754821777344
W = 3.380879163742065
H = 74.00595786344412
R = 0

#From there I will edit the values to see if we can determine whether the weather has the ecologically correct effect - change winds and temperatures to see how it changes rho 

In [ ]:
# I am going to simplify the weather variables, see if we can re do the analysis that I did manually on the powerpoint 


#Plug them in the windchill formula
#rho_pred = rho_max/(1 + exp( cT*T + cW*W + cH*H + cR*R - Tc ))

#with:
#rho_max = 12.83 1/m²
#T : temperature in K, W: windspeed in m/s, H: relative humidity in %, R: solar radiation in W/m²

#Then use rho_pred and area A to predict the number of individuals:
#count_pred = rho_pred * A


rho_pred_orig = rho_max/(1 + np.exp( (cT*T) + (cW*W) + (cH*H) + (cR*R) - Tc ))
print(rho_pred_orig)

6.766804908747927


In [9]:
rho_pred_paren = rho_max/(1 + np.exp( (cT*T) + (cW*W) + (cH*H) + (cR*R) - Tc ))
print(rho_pred_paren)

6.766804908747927


In [ ]:
# One example from Amanda Bay - first row of the dfRFD csv file 
# Windspeed         Temperature (K)     Humidity            Radiation
#3.380879163742065	247.7754821777344	74.00595786344412	0

T2 = 267.7754821777344 #added 20 degrees Kelvin
W = 3.380879163742065
H = 74.00595786344412
R = 0

#From there I will edit the values to see if we can determine whether the weather has the ecologically correct effect - change winds and temperatures to see how it changes rho 

In [13]:
rho_pred_20K = rho_max/(1 + np.exp( cT*T2 + cW*W + cH*H + cR*R - Tc ))
print(rho_pred_20K)

#I tested both parenthese and non parentheses and these make no difference. 

8.244632487432312


## This shows that when temperature increases, density increases. Which is not what we want, that does not make sense. 

In [ ]:
#Let's check windspeed too 
# One example from Amanda Bay - first row of the dfRFD csv file 
# Windspeed         Temperature (K)     Humidity            Radiation
#3.380879163742065	247.7754821777344	74.00595786344412	0

T = 247.7754821777344
W2 = 8.380879163742065 #changed from 3 to 8
H = 74.00595786344412
R = 0


rho_pred_8W = rho_max/(1 + np.exp( cT*T + cW*W2 + cH*H + cR*R - Tc ))
print(rho_pred_8W)

5.490856547990797


### This one here shows again, increase in windspeed decreases density 

Next up, is to break the equation down piece by piece. 

First we will do each individual part of the exponent, then we will do it step by step.

In [21]:
#rho_pred = rho_max/(1 + np.exp( cT*T + cW*W + cH*H + cR*R - Tc ))

print(cT*T)
print(cW*W)
print(cR*R)
print(cH*H)
print(Tc)

-5.908206372528078
0.27042300079107284
-0.0
0.6241662486202878
-4.903825


## Ok next up the exponent: 

The exponential function is e^x where e is a mathematical constant called Euler's number, approximately 2.718281. This value has a close mathematical relationship with pi and the slope of the curve e^x is equal to its value at every point. np.exp() calculates e^x for each value of x in your input array.

In [22]:
#rho_pred = rho_max/(1 + np.exp( cT*T + cW*W + cH*H + cR*R - Tc ))

np.exp(-5.908206372528078 + 0.27042300079107284 + 0.0 + 0.6241662486202878 + 4.903825)

np.float64(0.8960203778616039)

In [23]:
#rho_pred = rho_max/(1 + np.exp( cT*T + cW*W + cH*H + cR*R - Tc ))

1 + 0.8960203778616039

1.8960203778616038

In [ ]:
12.83/1.8960203778616038
#This matches with the full equation approach outlined above, indicating it is not a formatting error but appears to be a fundamental error with how the model is treating weather variables in relation to density

6.766804908747927

### Each part of this seems to be working, except for the affect of increasing temperature and decreasing density